# Lab 6 — Aiyagari, End to End

**ECON 282E · Session 6 · October 29, 2026**

Session 2 defined these equilibria. Session 6 computed them. This lab is the code.

| Part | What you build | Deck |
|---|---|---|
| 1 | the household problem by EGM, and the kink | 6.A1 |
| 2 | the Young lottery, the operator $Q$, and the invariant $\lambda$ | 6.A1 |
| 3 | clearing the market: bisection on $r$ | 6.A1 |
| 4 | accuracy: the refinement ladder and Euler residuals | 6.A1 |
| 5 | what persistence does | 6.A1 |
| 6 | a transition after a capital-tax reform | 6.A2 |
| 7 | welfare: the consumption-equivalent variation | 6.B3 |
| 8 | complete markets: two distributions, one aggregate | 6.B1 |
| 9 | Negishi: the weight fixed point | 6.B2 |

**Prerequisite:** L04 (interpolation, Rouwenhorst, root-finding). Everything is NumPy;
no GPU and no internet.

**Where the code comes from.** Part 0 inlines the *same* functions that generated
`tools/figures/s06_numbers.json`, which is where every number quoted in the Session 6
decks comes from. If your run disagrees with a lecture slide, one of you has a bug —
and it is worth finding out which.

**Timing.** Parts 1–5 run in a few minutes. Part 6 is the slow one (a couple of minutes).
Reduce `N_A` to 200 while you are experimenting.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# ---- the Session 6 calibration, quarterly (continuing S4 and S5) ----------
ALPHA = 0.33
BETA = 0.99
DELTA = 0.025
SIGMA = 2.0
SIGMA_RA = 1.0
RHO_Z = 0.95
SD_LOG_Z = 0.3
N_Z = 7
A_MIN = 0.0
A_MAX = 300.0
N_A = 400
CURV = 2.5
TOL_POL = 1e-10
TOL_DIST = 1e-12
TOL_R = 1e-07

print("quarterly:", ALPHA, BETA, DELTA, "| CRRA", SIGMA,
      "| earnings rho", RHO_Z, "sd log z", SD_LOG_Z)

---
## Part 0 — the solver

Read these functions before you run them. There are only six ideas here:
Rouwenhorst, a curved grid, EGM, the Young lottery, the operator $Q$, and bisection.

In [ ]:
# Inlined verbatim from tools/figures/s06_generate.py -- do not edit here.
def rouwenhorst(n, rho, sd_uncond):
    """Rouwenhorst (4.B3) for log z' = rho log z + eps with Var(log z) fixed.

    Returns (z levels normalised to mean one under the invariant, Pi, pi).
    """
    p = (1.0 + rho) / 2.0
    Pi = np.array([[p, 1 - p], [1 - p, p]])
    for k in range(3, n + 1):
        Pi_new = np.zeros((k, k))
        Pi_new[:-1, :-1] += p * Pi
        Pi_new[:-1, 1:] += (1 - p) * Pi
        Pi_new[1:, :-1] += (1 - p) * Pi
        Pi_new[1:, 1:] += p * Pi
        Pi_new[1:-1, :] /= 2.0
        Pi = Pi_new
    psi = sd_uncond * np.sqrt(n - 1)
    logz = np.linspace(-psi, psi, n)
    # invariant distribution of the Rouwenhorst chain is Binomial(n-1, 1/2)
    from math import comb
    pi = np.array([comb(n - 1, i) for i in range(n)], dtype=float) / 2.0 ** (n - 1)
    z = np.exp(logz)
    z = z / (pi @ z)                     # normalise mean effective labour to 1
    return z, Pi, pi

def make_grid(n, a_min, a_max, curv=CURV):
    """Exponentially spaced asset grid, dense near the borrowing constraint."""
    u = np.linspace(0.0, 1.0, n) ** curv
    return a_min + (a_max - a_min) * u

def prices_from_r(r, p=None):
    """Firm FOCs with L = 1 (mean effective labour normalised to one)."""
    kl = (ALPHA / (r + DELTA)) ** (1.0 / (1.0 - ALPHA))
    w = (1.0 - ALPHA) * kl ** ALPHA
    return kl, w                          # K demand per unit of labour, wage

def solve_egm(r, w, grid, z, Pi, tol=TOL_POL, max_iter=5000, c_init=None):
    """EGM / time iteration on the consumption policy over the joint state (a,z).

    Returns a'(a,z), c(a,z), iterations, final policy change.
    """
    na, nz = grid.size, z.size
    R = 1.0 + r
    coh = R * grid[:, None] + w * z[None, :]          # cash on hand
    c = coh - A_MIN if c_init is None else c_init.copy()
    c = np.maximum(c, 1e-10)
    it, diff = 0, np.inf
    for it in range(1, max_iter + 1):
        # marginal utility next period, evaluated ON the grid of a'
        Euc = (c ** (-SIGMA)) @ Pi.T                  # E[u'(c')|z] at a'=grid
        c_end = (BETA * R * Euc) ** (-1.0 / SIGMA)    # consumption today
        a_end = (c_end + grid[:, None] - w * z[None, :]) / R   # endogenous a
        c_new = np.empty_like(c)
        for q in range(nz):
            c_new[:, q] = np.interp(grid, a_end[:, q], c_end[:, q])
            # below the lowest endogenous point the constraint binds
            binds = grid < a_end[0, q]
            c_new[binds, q] = R * grid[binds] + w * z[q] - A_MIN
        c_new = np.maximum(c_new, 1e-10)
        diff = np.abs(c_new - c).max()
        c = c_new
        if diff < tol:
            break
    ap = np.clip(coh - c, A_MIN, grid[-1])
    c = coh - ap
    return ap, c, it, diff

def young_lottery(ap, grid):
    """Indices and upper weights for Young (2010) mass splitting."""
    a1 = np.clip(ap, grid[0], grid[-1])
    j = np.clip(np.searchsorted(grid, a1, side="right") - 1, 0, grid.size - 2)
    wt_hi = (a1 - grid[j]) / (grid[j + 1] - grid[j])
    return j, wt_hi

def push_forward(lam, j, wt_hi, Pi):
    """One application of the Markov operator Q on measures."""
    na, nz = lam.shape
    out = np.zeros_like(lam)
    for q in range(nz):
        tmp = np.zeros(na)
        np.add.at(tmp, j[:, q], lam[:, q] * (1.0 - wt_hi[:, q]))
        np.add.at(tmp, j[:, q] + 1, lam[:, q] * wt_hi[:, q])
        out += np.outer(tmp, Pi[q, :])
    return out

def stationary_dist(ap, grid, Pi, pi, tol=TOL_DIST, max_iter=100000):
    """Invariant lambda over (a,z) by iterating Q from a sensible start."""
    na, nz = ap.shape
    lam = np.zeros((na, nz))
    lam[0, :] = pi
    j, wt_hi = young_lottery(ap, grid)
    for it in range(1, max_iter + 1):
        lam_new = push_forward(lam, j, wt_hi, Pi)
        diff = np.abs(lam_new - lam).max()
        lam = lam_new
        if diff < tol:
            break
    return lam / lam.sum(), it, diff

def gini(grid, mass):
    """Gini of a distribution given on a grid of values."""
    m = mass / mass.sum()
    order = np.argsort(grid)
    x, m = np.asarray(grid)[order], m[order]
    S = np.cumsum(m * x)
    if S[-1] <= 0:
        return 0.0
    L = np.concatenate([[0.0], S / S[-1]])
    F = np.concatenate([[0.0], np.cumsum(m)])
    return float(1.0 - np.sum((F[1:] - F[:-1]) * (L[1:] + L[:-1])))

def aiyagari_at_r(r, grid, z, Pi, pi, c_init=None):
    """Household supply of capital at a candidate net return r."""
    kl, w = prices_from_r(r)
    ap, c, it_pol, d_pol = solve_egm(r, w, grid, z, Pi, c_init=c_init)
    lam, it_d, d_d = stationary_dist(ap, grid, Pi, pi)
    K_supply = float((lam.sum(axis=1) * grid).sum())
    return dict(r=r, w=w, K_demand=kl, K_supply=K_supply, ap=ap, c=c, lam=lam,
                it_pol=it_pol, it_dist=it_d)

def solve_r(grid, z, Pi, pi, r_lo=-0.004, r_hi=None, tol=TOL_R, verbose=False):
    """Bisection on r: excess demand d(r) = K_supply(r) - K_demand(r)."""
    if r_hi is None:
        r_hi = 1.0 / BETA - 1.0 - 1e-6        # r < 1/beta - 1 strictly
    f_lo = aiyagari_at_r(r_lo, grid, z, Pi, pi)
    d_lo = f_lo["K_supply"] - f_lo["K_demand"]
    f_hi = aiyagari_at_r(r_hi, grid, z, Pi, pi)
    d_hi = f_hi["K_supply"] - f_hi["K_demand"]
    if d_lo * d_hi > 0:
        raise RuntimeError(f"no sign change: d({r_lo})={d_lo}, d({r_hi})={d_hi}")
    n_iter = 0
    sol = f_lo
    while r_hi - r_lo > tol:
        n_iter += 1
        r_mid = 0.5 * (r_lo + r_hi)
        sol = aiyagari_at_r(r_mid, grid, z, Pi, pi, c_init=sol["c"])
        d_mid = sol["K_supply"] - sol["K_demand"]
        if verbose:
            print(f"    it {n_iter:2d}  r={r_mid: .6f}  d={d_mid: .6f}")
        if d_lo * d_mid <= 0:
            r_hi, d_hi = r_mid, d_mid
        else:
            r_lo, d_lo = r_mid, d_mid
    sol["bisection_iters"] = n_iter
    return sol

print("loaded")

---
## Part 1 — the household problem, and the kink

Fix a return `r` and solve one household problem. Nothing is in equilibrium yet.

In [ ]:
z, Pi, pi = rouwenhorst(N_Z, RHO_Z, SD_LOG_Z)
grid = make_grid(N_A, A_MIN, A_MAX)
print("z =", np.round(z, 4))
print("E[z] =", pi @ z, "  sd(log z) =",
      np.sqrt(pi @ (np.log(z) - pi @ np.log(z))**2))

r_try = 0.008
kl, w = prices_from_r(r_try)
t0 = time.time()
ap, c, n_it, diff = solve_egm(r_try, w, grid, z, Pi)
print("EGM: %d iterations, change %.2e, %.2fs" % (n_it, diff, time.time()-t0))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sel = grid <= 60
for q in (0, N_Z//2, N_Z-1):
    ax[0].plot(grid[sel], ap[sel, q], label="z = %.2f" % z[q])
ax[0].plot(grid[sel], grid[sel], "k--", lw=0.8, label="45 degrees")
ax[0].set_xlabel("a"); ax[0].set_ylabel("a'"); ax[0].legend(fontsize=7)
ax[0].set_title("saving policy")
for q in (0, N_Z-1):
    ax[1].plot(grid[sel], c[sel, q], label="z = %.2f" % z[q])
ax[1].set_xlabel("a"); ax[1].set_ylabel("c"); ax[1].legend(fontsize=7)
ax[1].set_title("consumption policy")
plt.tight_layout(); plt.show()

at_limit = ap <= A_MIN + 1e-9
print("grid nodes at the borrowing limit:", at_limit.sum(), "of", ap.size)

**Exercise 1.** The kink in `a'(a, z)` is where the borrowing constraint stops binding.
Locate it for the lowest `z` and report the asset level. Then re-solve with `CURV = 1.0`
(a uniform grid) at the same `N_A` and report how the location moves. Which discretization
resolves the kink better, and why does grid curvature matter *here* specifically?

**Exercise 2.** Verify that EGM never solves an optimization problem. Instrument
`solve_egm` to count calls to any root-finder or maximizer. Then explain in two sentences
what replaced the search.

---
## Part 2 — from a policy to a population

The policy sends a household off the grid. The Young lottery decides where its mass goes.

In [ ]:
j, wt = young_lottery(ap, grid)
q_show = 0
print("a' = %.4f lands between a[%d] = %.4f and a[%d] = %.4f, weight %.4f"
      % (ap[50, q_show], j[50, q_show], grid[j[50, q_show]],
         j[50, q_show]+1, grid[j[50, q_show]+1], wt[50, q_show]))

t0 = time.time()
lam, it_d, d_d = stationary_dist(ap, grid, Pi, pi)
print("lambda: %d applications of Q, change %.2e, %.2fs"
      % (it_d, d_d, time.time()-t0))
print("mass conservation error: %.2e" % abs(lam.sum() - 1.0))
print("mass at the top of the grid: %.2e" % lam.sum(axis=1)[-1])

lam_a = lam.sum(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
sel = grid <= 120
ax[0].plot(grid[sel], lam_a[sel])
ax[0].set_xlabel("a"); ax[0].set_title("marginal distribution of assets")
ax[1].plot(grid[sel], np.cumsum(lam_a)[sel])
ax[1].set_xlabel("a"); ax[1].set_title("CDF")
plt.tight_layout(); plt.show()
print("wealth Gini at this r:", round(gini(grid, lam_a), 4))

**Exercise 3.** Replace the lottery with nearest-grid-point rounding. Total mass is still
one, so the distribution is still a distribution. Show that the mean of `a` under
$\lambda$ is nevertheless biased, at `N_A = 200` and at `N_A = 800`. Roughly what power
of the grid spacing is the bias?

**Exercise 4.** Get $\lambda$ three ways: the power iteration above, a dense solve of
$(Q' - I)\lambda = 0$ with a normalization row, and a sparse eigenvector at eigenvalue 1
(`scipy.sparse.linalg.eigs`). Report wall time and the largest disagreement between them.
At what `N_A` does the dense solve stop being competitive?

---
## Part 3 — closing the model

Now impose market clearing. Excess demand is
`d(r) = (household supply) - (firm demand)`, and **both** terms must depend on `r`.

In [ ]:
rs = np.linspace(-0.002, 1.0/BETA - 1.0 - 2e-4, 12)
Ks, Kd, warm = [], [], None
for rr in rs:
    f = aiyagari_at_r(rr, grid, z, Pi, pi, c_init=warm)
    warm = f["c"]
    Ks.append(f["K_supply"]); Kd.append(f["K_demand"])

plt.figure(figsize=(5.4, 3.8))
plt.plot(Ks, rs, label="supply A(r)")
plt.plot(Kd, rs, label="demand K^d(r)")
plt.axhline(1.0/BETA - 1.0, color="k", ls="--", lw=0.8, label="1/beta - 1")
plt.xlabel("capital"); plt.ylabel("r (quarterly)"); plt.legend(fontsize=8)
plt.xlim(0, 56); plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
sol = solve_r(grid, z, Pi, pi, verbose=True)
print("\nr* = %.6f (quarterly)  =  %.4f%% annual"
      % (sol["r"], 100*((1+sol["r"])**4 - 1)))
print("rbar = 1/beta - 1 = %.6f   wedge = %.1f basis points"
      % (1/BETA - 1, 1e4*((1/BETA - 1) - sol["r"])))
print("K* = %.4f   K/Y (annual) = %.3f"
      % (sol["K_supply"], sol["K_supply"]/(4*sol["K_supply"]**ALPHA)))
lam = sol["lam"]; lam_a = lam.sum(axis=1)
print("wealth Gini = %.4f   at the limit = %.4f%%"
      % (gini(grid, lam_a), 100*lam[0, :].sum()))
print("solved in %.1fs" % (time.time()-t0))

**Exercise 5.** The lecture reports $r^* = 0.00804$, a wedge of 20.6 basis points, and a
wealth Gini of 0.469. Reproduce all three. If you do not, the most likely culprits are
`N_A`, `A_MAX` and `CURV` — show which one moves your answer most.

**Exercise 6.** Replace bisection with Newton on a finite-difference derivative of `d(r)`.
Count household solves for each method. Then explain why bisection is still the
recommendation: look at what happens to the *set of constrained nodes* as `r` moves, and
what that does to the smoothness of `A(r)`.

**Exercise 7 (trap).** Someone defines excess demand as `d(r) = K(r) - integral a dlambda`
where `K(r)` is set to household *supply*. Show that this is identically zero in a
stationary equilibrium, so any `r` "clears" it. What would you see in the output?

---
## Part 4 — do you believe it?

Five checks. Only the fourth is expensive, and it is the one that most often moves an answer.

In [ ]:
rows = []
for n_a in (100, 200, 400):
    g2 = make_grid(n_a, A_MIN, A_MAX)
    t0 = time.time()
    s2 = solve_r(g2, z, Pi, pi)
    rows.append((n_a, s2["r"], s2["K_supply"],
                 gini(g2, s2["lam"].sum(axis=1)), time.time()-t0))
print(" n_a        r*          K*      Gini     sec")
for a, b, cc, d, e in rows:
    print("%4d  %.6f  %8.4f  %.4f  %6.1f" % (a, b, cc, d, e))

In [ ]:
# unit-free Euler residual, where the constraint is slack and a' is not clipped
ap, c = sol["ap"], sol["c"]
R = 1.0 + sol["r"]
Euc = np.zeros_like(c)
for q in range(N_Z):
    cp = np.empty((grid.size, N_Z))
    for qq in range(N_Z):
        cp[:, qq] = np.interp(ap[:, q], grid, c[:, qq])
    Euc[:, q] = (cp ** (-SIGMA)) @ Pi[q, :]
resid = np.abs(1.0 - (BETA*R*Euc) ** (-1.0/SIGMA) / c)
off = (ap > A_MIN + 1e-9) & (ap < grid[-1]*(1-1e-9))
v = resid[off]
for p in (50, 90, 99, 100):
    print("p%-4s  %.3e   (log10 %.2f)"
          % (p, np.percentile(v, p), np.log10(np.percentile(v, p))))
print("nodes above 1e-2:", int((v > 1e-2).sum()), "of", v.size)

**Exercise 8.** The worst few nodes are not where you expect. Find the `(a, z)` at which
the residual is largest and report the asset level. Then report how much of $\lambda$ sits
there. Argue, in two sentences, whether the median or the maximum is the honest summary of
this solution — and what you would have to change for the maximum to become the right one.

---
## Part 5 — persistence, with dispersion held fixed

In [ ]:
out = []
for rho in (0.90, 0.95, 0.99):
    zz, PP, pp = rouwenhorst(N_Z, rho, SD_LOG_Z)     # same unconditional sd
    s3 = solve_r(grid, zz, PP, pp)
    la = s3["lam"]
    out.append((rho, SD_LOG_Z*np.sqrt(1-rho**2), 100*((1+s3["r"])**4 - 1),
                gini(grid, la.sum(axis=1)), 100*la[0, :].sum()))
print(" rho   sigma_eps   r* ann     Gini   at limit")
for a, b, cc, d, e in out:
    print("%.2f    %.4f     %.3f%%   %.4f   %6.2f%%" % (a, b, cc, d, e))

**Exercise 9.** The lecture reports the constrained share rising from 0.51% to 12.98% as
$\rho$ goes from 0.90 to 0.99, with the dispersion of $\log z$ held fixed. Reproduce it.
Then redo the sweep holding the *conditional* standard deviation fixed instead. Does the
lecture's claim survive? State precisely what changed.

---
## Part 6 — a transition

A permanent 20% capital-income tax, rebated lump sum, announced at $t = 0$.
Backward on decisions, forward on the distribution, damped update in between.

In [ ]:
def solve_egm_with_transfer(rnet, w, transfer, grid, z, Pi, tol=TOL_POL,
                            max_iter=5000, c_init=None):
    """EGM with a lump-sum transfer in the budget constraint."""
    na, nz = grid.size, z.size
    R = 1.0 + rnet
    coh = R * grid[:, None] + w * z[None, :] + transfer
    c = coh - A_MIN if c_init is None else c_init.copy()
    c = np.maximum(c, 1e-10)
    it, diff = 0, np.inf
    for it in range(1, max_iter + 1):
        Euc = (c ** (-SIGMA)) @ Pi.T
        c_end = (BETA * R * Euc) ** (-1.0 / SIGMA)
        a_end = (c_end + grid[:, None] - w * z[None, :] - transfer) / R
        c_new = np.empty_like(c)
        for q in range(nz):
            c_new[:, q] = np.interp(grid, a_end[:, q], c_end[:, q])
            binds = grid < a_end[0, q]
            c_new[binds, q] = R * grid[binds] + w * z[q] + transfer - A_MIN
        c_new = np.maximum(c_new, 1e-10)
        diff = np.abs(c_new - c).max()
        c = c_new
        if diff < tol:
            break
    ap = np.clip(coh - c, A_MIN, grid[-1])
    return ap, coh - ap, it, diff

def _backward_policies(path_rnet, path_w, path_T, grid, z, Pi, c_final):
    """Backward pass: consumption policies along a transition of length T."""
    T = len(path_rnet) - 1
    na, nz = grid.size, z.size
    c_path = [None] * (T + 1)
    ap_path = [None] * (T + 1)
    c_path[T] = c_final
    for t in range(T - 1, -1, -1):
        Rn = 1.0 + path_rnet[t + 1]
        Euc = (c_path[t + 1] ** (-SIGMA)) @ Pi.T
        c_end = (BETA * Rn * Euc) ** (-1.0 / SIGMA)
        Rt = 1.0 + path_rnet[t]
        a_end = (c_end + grid[:, None] - path_w[t] * z[None, :] - path_T[t]) / Rt
        c_t = np.empty((na, nz))
        for q in range(nz):
            c_t[:, q] = np.interp(grid, a_end[:, q], c_end[:, q])
            binds = grid < a_end[0, q]
            c_t[binds, q] = (Rt * grid[binds] + path_w[t] * z[q]
                             + path_T[t] - A_MIN)
        c_t = np.maximum(c_t, 1e-10)
        coh = Rt * grid[:, None] + path_w[t] * z[None, :] + path_T[t]
        c_path[t] = c_t
        ap_path[t] = np.clip(coh - c_t, A_MIN, grid[-1])
    return c_path, ap_path

def _value_ss(c, ap, grid, Pi, tol=1e-11, max_iter=20000):
    """Value function of a stationary policy."""
    nz = Pi.shape[0]
    V = (c ** (1.0 - SIGMA) / (1.0 - SIGMA)) / (1.0 - BETA)
    u = c ** (1.0 - SIGMA) / (1.0 - SIGMA)
    for _ in range(max_iter):
        EV = V @ Pi.T
        cont = np.empty_like(EV)
        for q in range(nz):
            cont[:, q] = np.interp(ap[:, q], grid, EV[:, q])
        V_new = u + BETA * cont
        if np.abs(V_new - V).max() < tol:
            V = V_new
            break
        V = V_new
    return V

def _values_along(c_path, ap_path, grid, Pi, V_final):
    """Backward pass for V, needed for the consumption-equivalent variation."""
    T = len(c_path) - 1
    V = [None] * (T + 1)
    V[T] = V_final
    nz = Pi.shape[0]
    for t in range(T - 1, -1, -1):
        EV = V[t + 1] @ Pi.T                     # E[V_{t+1}(a',z')|z] on grid
        cont = np.empty_like(EV)
        for q in range(nz):
            cont[:, q] = np.interp(ap_path[t][:, q], grid, EV[:, q])
        u = c_path[t] ** (1.0 - SIGMA) / (1.0 - SIGMA)
        V[t] = u + BETA * cont
    return V

def _halflife(path, x0, x1):
    """First period at which the path has covered half the total change."""
    if abs(x1 - x0) < 1e-14:
        return float("nan")
    frac = (path - x0) / (x1 - x0)
    idx = np.where(frac >= 0.5)[0]
    return float(idx[0]) if idx.size else float("nan")

print("loaded the transition machinery")

In [ ]:
TAU_K, T = 0.20, 200

def ss_with_tax(tau_k, n_bisect=45):
    lo, hi, s = 0.0005, 1.0/BETA - 1.0 - 1e-6, None
    for _ in range(n_bisect):
        mid = 0.5*(lo + hi)
        kl, w = prices_from_r(mid)
        ap_, c_, _, _ = solve_egm_with_transfer(
            (1-tau_k)*mid, w, tau_k*mid*kl, grid, z, Pi,
            c_init=None if s is None else s["c"])
        la, _, _ = stationary_dist(ap_, grid, Pi, pi)
        Ks = float((la.sum(axis=1)*grid).sum())
        s = dict(r=mid, w=w, ap=ap_, c=c_, lam=la, K_supply=Ks, K_demand=kl)
        hi, lo = (mid, lo) if Ks - kl > 0 else (hi, mid)
    return s

t0 = time.time()
ss0, ss1 = ss_with_tax(0.0), ss_with_tax(TAU_K)
print("old K = %.4f   new K = %.4f   (%.2f%%)   [%.0fs]"
      % (ss0["K_supply"], ss1["K_supply"],
         100*(ss1["K_supply"]/ss0["K_supply"] - 1), time.time()-t0))

In [ ]:
K0, K1 = ss0["K_supply"], ss1["K_supply"]
Kpath = K0 + (K1-K0)*(1 - np.exp(-np.arange(T+1)/25.0)); Kpath[0] = K0
damp = 0.30
for outer in range(1, 201):
    r_pre = ALPHA*Kpath**(ALPHA-1) - DELTA
    w_p   = (1-ALPHA)*Kpath**ALPHA
    rnet  = (1-TAU_K)*r_pre; rnet[0] = ss0["r"]
    reb   = TAU_K*r_pre*Kpath
    c_path, ap_path = _backward_policies(rnet, w_p, reb, grid, z, Pi, ss1["c"])
    lam_t, K_new = ss0["lam"].copy(), np.empty(T+1); K_new[0] = K0
    for t in range(T):
        jj, ww = young_lottery(ap_path[t], grid)
        lam_t = push_forward(lam_t, jj, ww, Pi)
        K_new[t+1] = float((lam_t.sum(axis=1)*grid).sum())
    gap = float(np.abs(K_new - Kpath).max())
    Kpath = (1-damp)*Kpath + damp*K_new
    if gap < 1e-6:
        break
print("converged in %d outer iterations, max|A_t - K_t| = %.2e" % (outer, gap))
print("lambda_T distance to the new steady state (L1): %.4f"
      % np.abs(lam_t - ss1["lam"]).sum())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(Kpath); ax[0].axhline(K0, ls="--", c="k", lw=0.8)
ax[0].axhline(K1, ls=":", c="k", lw=0.8); ax[0].set_title("capital")
ax[1].plot(ALPHA*Kpath**(ALPHA-1) - DELTA); ax[1].set_title("pre-tax r")
for a_ in ax: a_.set_xlabel("quarters")
plt.tight_layout(); plt.show()

**Exercise 10.** Set `T = 80` and then `T = 400`, and report the aggregate CEV of Part 7
each time. How large must `T` be before the third decimal stops moving? Explain what the
`lambda_T` distance printed above is telling you, and why it is the right thing to check.

**Exercise 11.** Remove the damping (`damp = 1.0`). Describe what happens and why.

---
## Part 7 — who wins

In [ ]:
V0 = _value_ss(ss0["c"], ss0["ap"], grid, Pi)
V1 = _value_ss(ss1["c"], ss1["ap"], grid, Pi)
Vtr = _values_along(c_path, ap_path, grid, Pi, V1)
g_tr = (Vtr[0]/V0)**(1.0/(1.0-SIGMA)) - 1.0      # along the transition
g_ss = (V1  /V0)**(1.0/(1.0-SIGMA)) - 1.0        # steady states only
w0 = ss0["lam"]/ss0["lam"].sum()
print("aggregate CEV, transition   : %+.4f%%" % (100*(g_tr*w0).sum()))
print("aggregate CEV, steady states: %+.4f%%" % (100*(g_ss*w0).sum()))
print("share worse off             : %.1f%%"  % (100*w0[g_tr < 0].sum()))
print("share changing sign         : %.1f%%"
      % (100*w0[np.sign(g_tr) != np.sign(g_ss)].sum()))

**Exercise 12.** Report the CEV by decile of the *initial* wealth distribution and say who
gains. Then explain the sign of the gap between the two aggregate numbers above — and why
it runs the opposite way from the Auerbach–Kotlikoff warning you met in 2.B2.

---
## Part 8 — the case where none of this was necessary

In [ ]:
def _k_ss_ra():
    return (ALPHA / (1.0 / BETA - 1.0 + DELTA)) ** (1.0 / (1.0 - ALPHA))

def _shoot(k0, c0, T, kss, lo_frac=0.30, hi_frac=1.50):
    """Forward-integrate the RA growth model from (k0,c0), log utility.

    The corridor is deliberately tight: the saddle path is unstable, so any
    c0 that is not exactly right eventually leaves it.  Which side it leaves
    on is what the bisection reads.
    """
    k = np.empty(T + 1); c = np.empty(T + 1)
    k[0], c[0] = k0, c0
    for t in range(T):
        k[t + 1] = k[t] ** ALPHA + (1.0 - DELTA) * k[t] - c[t]
        if k[t + 1] <= lo_frac * kss:
            return k[:t + 2], c[:t + 1], t + 1, "low"
        if k[t + 1] >= hi_frac * kss:
            return k[:t + 2], c[:t + 1], t + 1, "high"
        R = ALPHA * k[t + 1] ** (ALPHA - 1.0) + 1.0 - DELTA
        c[t + 1] = c[t] * (BETA * R) ** (1.0 / SIGMA_RA)
    return k, c, T, "stayed"

def _planner_path(thetas, sigmas, ns, k0, T=400):
    """Planner allocation with Pareto weights, by shooting on the multiplier.

    theta_i u_i'(c_it) = mu_t, resources: sum_i n_i c_it + k' = f(k)+(1-d)k.
    Returns (k, C, mu) along the path.
    """
    kss = _k_ss_ra()

    def C_of_mu(mu):
        return sum(n * (th / mu) ** (1.0 / s)
                   for th, s, n in zip(thetas, sigmas, ns))

    def integrate(mu0):
        k = np.empty(T + 1); mu = np.empty(T + 1); C = np.empty(T + 1)
        k[0], mu[0] = k0, mu0
        for t in range(T):
            C[t] = C_of_mu(mu[t])
            k[t + 1] = k[t] ** ALPHA + (1.0 - DELTA) * k[t] - C[t]
            if k[t + 1] <= 0.30 * kss or k[t + 1] >= 1.50 * kss:
                return k[:t + 2], C[:t + 1], mu[:t + 1], t + 1, (
                    "low" if k[t + 1] <= 0.30 * kss else "high")
            R = ALPHA * k[t + 1] ** (ALPHA - 1.0) + 1.0 - DELTA
            mu[t + 1] = mu[t] / (BETA * R)
        C[T] = C_of_mu(mu[T])
        return k, C, mu, T, "stayed"

    lo, hi = 1e-12, 1e6
    for _ in range(300):
        mid = np.sqrt(lo * hi)
        _, _, _, _, why = integrate(mid)
        if why == "high":          # mu too high -> C too low -> k explodes
            hi = mid
        elif why == "low":
            lo = mid
        else:
            break
        if hi / lo < 1 + 1e-15:
            break
    mu0 = np.sqrt(lo * hi)
    return integrate(mu0)

kss = _k_ss_ra(); k0 = 0.5*kss
paths = []
for th in ([0.5, 0.5], [0.2, 0.8], [0.05, 0.95]):
    kk, CC, mu, _, _ = _planner_path(th, [SIGMA_RA, SIGMA_RA], [0.5, 0.5], k0, 300)
    paths.append(kk[:150])
print("common curvature, max |K^A - K^B| over 150 quarters: %.3e"
      % max(np.abs(paths[0]-q).max() for q in paths[1:]))

paths_h = []
for th in ([0.5, 0.5], [0.2, 0.8], [0.05, 0.95]):
    kk, CC, mu, _, _ = _planner_path(th, [1.0, 5.0], [0.5, 0.5], k0, 300)
    paths_h.append(kk[:150])
print("mixed curvature,  max |K^A - K^B|:                   %.3e"
      % max(np.abs(paths_h[0]-q).max() for q in paths_h[1:]))

**Exercise 13.** The first number is zero to machine precision and the second is not.
Say exactly which Gorman condition the second experiment breaks, and predict — before
running it — what happens if instead you give the two households the same curvature but
different *discount factors*. Then run it.

---
## Part 9 — Negishi

In [ ]:
def _ra_path(k0, T=400):
    """The representative-agent transition, and Arrow-Debreu prices along it.

    Log utility, so q_{t+1}/q_t -> beta once the economy has settled; the
    infinite tail beyond T is added in closed form rather than truncated.
    """
    kss = _k_ss_ra()
    css = kss ** ALPHA - DELTA * kss
    lo, hi = 1e-6, k0 ** ALPHA + (1.0 - DELTA) * k0 - 1e-8
    for _ in range(240):
        mid = 0.5 * (lo + hi)
        _, _, _, why = _shoot(k0, mid, T, kss)
        if why == "high":
            lo = mid
        elif why == "low":
            hi = mid
        else:
            break
        if (hi - lo) <= 2.3e-16 * max(abs(hi), 1.0):
            break
    c0 = 0.5 * (lo + hi)
    k, c, _, _ = _shoot(k0, c0, T, kss)
    n = min(len(k), len(c))
    k, c = k[:n], c[:n]
    q = BETA ** np.arange(n) * (c[0] / c)          # q_0 = 1
    r = ALPHA * k ** (ALPHA - 1.0) - DELTA
    w = (1.0 - ALPHA) * k ** ALPHA
    # Present values are truncated where the shot is still accurate and the
    # infinite tail is added from the steady state.  The tail carries a factor
    # beta/(1-beta) = 99, so it must not be taken from the drifting end of the
    # path: q_T there is already contaminated by the unstable root.
    Tp = min(250, n - 1)
    wss = (1.0 - ALPHA) * kss ** ALPHA
    tail = BETA / (1.0 - BETA)
    PV_w = float(q[:Tp + 1] @ w[:Tp + 1]) + float(q[Tp] * wss * tail)
    PV_C = float(q[:Tp + 1] @ c[:Tp + 1]) + float(q[Tp] * css * tail)
    return dict(k=k, c=c, q=q, r=r, w=w, n=n, kss=kss, css=css, T_pv=Tp,
                PV_w=PV_w, PV_C=PV_C)

ra = _ra_path(0.5*_k_ss_ra(), T=400)
n_i = np.array([0.5, 0.5])
k_i0 = np.array([0.25, 1.75])*ra["k"][0]
W = (1.0 + ra["r"][0])*k_i0 + ra["PV_w"]
print("capital ratio      : %.2f" % (k_i0[1]/k_i0[0]))
print("wealth ratio       : %.4f" % (W[1]/W[0]))
print("human wealth share : %.1f%%" % (100*ra["PV_w"]/float(n_i @ W)))
print("Walras check       : %.2e"
      % (abs(float(n_i @ W) - ra["PV_C"]) / ra["PV_C"]))

lo, hi = 1e-8, 1.0/n_i[0] - 1e-8
for it in range(1, 121):
    mid = 0.5*(lo + hi)
    h1 = mid*ra["PV_C"] - W[0]
    hi, lo = (mid, lo) if h1 > 0 else (hi, mid)
    if abs(h1) < 1e-13*ra["PV_C"]:
        break
print("theta_1 = %.6f in %d bisections; closed form %.6f"
      % (0.5*(lo+hi), it, W[0]/float(n_i @ W)))

**Exercise 14.** Solve the same two-household economy the other way: guess the price path,
compute each household's demand, and iterate until markets clear at every date. Report the
number of unknowns and the iterations each method needs. Then say what would happen to each
method if you added a borrowing constraint — and which one stops being *correct* rather
than merely slow.

---
## What to take away

1. **Aiyagari is three nested fixed points.** A dynamic program inside an invariant
   distribution inside a scalar price. Every outer residual evaluation costs both inner solves.
2. **EGM removes the inner search** by gridding tomorrow's assets and inverting the Euler
   equation. The borrowing constraint becomes a comparison, not a solver.
3. **The Young lottery is not interpolation for its own sake.** It is what lets you push a
   distribution forward deterministically, and it is why none of this needs simulation noise.
4. **Bisection beats Newton here** because the set of constrained households changes
   discretely with `r`, so `A(r)` is only piecewise smooth.
5. **Report the median residual and say where the maximum is.** A single bad node at the
   edge of a grid is not the same kind of problem as a bad node in the middle of the mass.
6. **Persistence, not variance, creates constrained households** — and the constrained
   households are the entire reason this model is not the representative-agent model.
7. **A transition needs both steady states and a check that it arrived.** If `lambda_T` is
   far from the new stationary distribution, `T` was too short and every welfare number is
   contaminated.
8. **Steady-state welfare comparisons are not welfare comparisons.** They can differ in
   magnitude, and for a third of households here they differ in sign.
9. **None of this work is necessary under complete markets with preferences that
   aggregate** — and that is a much smaller set of models than the habit of writing down a
   representative agent suggests.